# 🧠 EEG Motor Action Classification
## Notebook 3 — Results Analysis & Model Comparison

---

This notebook loads saved training results and produces publication-quality
comparison plots and statistical analysis across the three models.

### Contents
1. Load saved results  
2. Metrics comparison table  
3. Confusion matrices (side by side)  
4. ROC & Precision-Recall curves  
5. Per-class F1 heatmap  
6. Parameter count vs. accuracy trade-off  
7. Error analysis — most confused class pairs  
8. Conclusions & recommendations

---
## 0. Setup

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from config import CLASS_NAMES, N_CLASSES, RESULTS_DIR
from src.utils.metrics import (
    evaluate, per_class_metrics, build_comparison_table
)
from src.utils.visualization import (
    plot_confusion_matrix, plot_roc_curves,
    plot_precision_recall, plot_model_comparison,
    plot_training_history,
)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print('✅ Setup complete')

---
## 1. Load Saved Results

> Run Notebook 02 first to generate the results files.
> This notebook can also consume the `all_results` dict directly if run in the same kernel.

In [ ]:
# If running independently, load from CSV
comparison_path = RESULTS_DIR / '02_model_comparison.csv'

if comparison_path.exists():
    comp_df = pd.read_csv(comparison_path, index_col=0)
    print('Loaded comparison table from file:')
    display(comp_df)
else:
    print('⚠️  No saved results found. Run Notebook 02 first.')
    print('   Or run train.py --model all')
    comp_df = None

In [ ]:
# Load per-model metrics from json
model_dirs = {'EEGNet': 'EEGNet', 'ResNet1D': 'ResNet1D_Lite', 'BiLSTM': 'BiLSTM_standard'}
saved_metrics = {}
saved_history = {}

for display_name, dir_name in model_dirs.items():
    metrics_file = RESULTS_DIR / dir_name / 'metrics.json'
    history_file = RESULTS_DIR / dir_name / 'history.json'
    
    if metrics_file.exists():
        with open(metrics_file) as f:
            saved_metrics[display_name] = json.load(f)
        print(f'  ✅ Loaded {display_name} metrics')
    else:
        print(f'  ⚠️  {display_name} metrics not found — run training first')
    
    if history_file.exists():
        with open(history_file) as f:
            saved_history[display_name] = json.load(f)

print(f'\nLoaded metrics for: {list(saved_metrics.keys())}')

---
## 2. Metrics Comparison Table

In [ ]:
if saved_metrics:
    table = build_comparison_table(saved_metrics)
    
    # Styled display
    styled = (
        table.style
        .highlight_max(axis=0, color='#d4edda', props='font-weight: bold;')
        .highlight_min(axis=0, color='#f8d7da')
        .format('{:.4f}')
        .set_caption('Model Comparison — Test Set (bold = best, red = worst)')
    )
    display(styled)
    
    table.to_csv(RESULTS_DIR / '03_final_comparison.csv')
    print('\nSaved to results/03_final_comparison.csv')
else:
    print('No metrics available. Run Notebook 02 first.')

---
## 3. Metrics Bar Chart

In [ ]:
if saved_metrics:
    metrics_to_plot = ['accuracy', 'f1_macro', 'cohen_kappa', 'auc_macro', 'mcc']
    n_met  = len(metrics_to_plot)
    models = list(saved_metrics.keys())
    colors = ['steelblue', 'coral', 'seagreen']
    
    fig, axes = plt.subplots(1, n_met, figsize=(18, 5))
    
    for ax, metric in zip(axes, metrics_to_plot):
        vals  = [saved_metrics[m].get(metric, 0) for m in models]
        bars  = ax.bar(models, vals, color=colors[:len(models)],
                       edgecolor='k', linewidth=0.6)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9,
                    fontweight='bold')
        ax.set_ylim(0, 1.12)
        ax.set_title(metric.replace('_',' ').title(), fontsize=11)
        ax.set_ylabel('Score')
        ax.spines[['top','right']].set_visible(False)
        ax.tick_params(axis='x', rotation=15)
    
    fig.suptitle('Model Comparison — Test Set Metrics',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/03_metrics_comparison.png', bbox_inches='tight', dpi=150)
    plt.show()

---
## 4. Training Curves Overlay

In [ ]:
if saved_history:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    styles = [
        ('EEGNet',   'steelblue', '-'),
        ('ResNet1D', 'coral',     '--'),
        ('BiLSTM',   'seagreen',  ':'),
    ]
    
    for name, color, ls in styles:
        if name not in saved_history:
            continue
        hist = saved_history[name]
        ep   = range(1, len(hist.get('loss', [])) + 1)
        if 'val_loss' in hist:
            axes[0].plot(ep, hist['val_loss'],     color=color, ls=ls,
                         label=name, linewidth=1.8)
        if 'val_accuracy' in hist:
            axes[1].plot(ep, hist['val_accuracy'], color=color, ls=ls,
                         label=name, linewidth=1.8)
    
    labels = ['Validation Loss', 'Validation Accuracy']
    for ax, lbl in zip(axes, labels):
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_title(lbl, fontsize=12)
        ax.legend(fontsize=10)
        ax.spines[['top','right']].set_visible(False)
    
    fig.suptitle('Validation Curves — All Models', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/03_validation_curves.png', bbox_inches='tight', dpi=150)
    plt.show()

---
## 5. Per-Class F1 Heatmap

> Loads per-class CSV files saved during training.

In [ ]:
f1_data = {}
for display_name, dir_name in model_dirs.items():
    pc_file = RESULTS_DIR / dir_name / 'per_class.csv'
    if pc_file.exists():
        df_pc = pd.read_csv(pc_file, index_col=0)
        f1_data[display_name] = df_pc['f1-score']

if f1_data:
    f1_df = pd.DataFrame(f1_data)    # rows = classes, cols = models
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(
        f1_df,
        ax=ax, annot=True, fmt='.3f',
        cmap='RdYlGn', vmin=0, vmax=1,
        linewidths=0.5, linecolor='white',
        cbar_kws={'label': 'F1-Score'},
    )
    ax.set_title('Per-Class F1-Score by Model', fontsize=13, fontweight='bold')
    ax.set_xlabel('Model')
    ax.set_ylabel('Class')
    plt.tight_layout()
    plt.savefig('../results/03_per_class_f1_heatmap.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print('No per-class data found. Run Notebook 02 first.')

---
## 6. Parameters vs. Accuracy Trade-off

In [ ]:
# Approximate parameter counts (from model.count_params())
model_params = {
    'EEGNet':   4_200,
    'ResNet1D': 500_000,    # Lite variant
    'BiLSTM':   370_000,
}

if saved_metrics:
    accs  = [saved_metrics[m].get('accuracy', 0) for m in model_params]
    f1s   = [saved_metrics[m].get('f1_macro', 0) for m in model_params]
    names = list(model_params.keys())
    params = list(model_params.values())
    colors = ['steelblue', 'coral', 'seagreen']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for ax, scores, ylabel in zip(axes, [accs, f1s], ['Accuracy', 'F1 Macro']):
        for name, p, s, c in zip(names, params, scores, colors):
            ax.scatter(p, s, s=200, color=c, zorder=5, label=name)
            ax.annotate(name, (p, s), textcoords='offset points',
                        xytext=(10, 5), fontsize=10, color=c, fontweight='bold')
        ax.set_xscale('log')
        ax.set_xlabel('Model Parameters (log scale)', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(f'{ylabel} vs. Model Size', fontsize=12)
        ax.legend(fontsize=9)
        ax.spines[['top','right']].set_visible(False)
    
    fig.suptitle('Efficiency Trade-off: Parameters vs. Performance',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/03_params_vs_accuracy.png', bbox_inches='tight', dpi=150)
    plt.show()

---
## 7. Error Analysis — Most Confused Class Pairs

In [ ]:
# This cell requires all_results from Notebook 02 in memory.
# If running independently, skip or re-run training.

try:
    from sklearn.metrics import confusion_matrix
    
    for model_name, res in all_results.items():
        cm = confusion_matrix(res['y_test'], res['y_pred'])
        
        # Find top-5 most common misclassifications
        errors = []
        for true_cls in range(N_CLASSES):
            for pred_cls in range(N_CLASSES):
                if true_cls != pred_cls and cm[true_cls, pred_cls] > 0:
                    errors.append((cm[true_cls, pred_cls],
                                   CLASS_NAMES[true_cls],
                                   CLASS_NAMES[pred_cls]))
        
        errors.sort(reverse=True)
        print(f'\n{model_name} — Top-5 misclassifications:')
        print(f'  {"True":8s} → {"Predicted":8s}  Count')
        for cnt, true_c, pred_c in errors[:5]:
            print(f'  {true_c:8s} → {pred_c:8s}  {cnt}')

except NameError:
    print('`all_results` not in memory. Run Notebook 02 first.')

---
## 8. Side-by-Side Confusion Matrices

In [ ]:
try:
    from sklearn.metrics import confusion_matrix
    
    n_models = len(all_results)
    fig, axes = plt.subplots(1, n_models, figsize=(7*n_models, 6))
    if n_models == 1:
        axes = [axes]
    
    for ax, (model_name, res) in zip(axes, all_results.items()):
        cm = confusion_matrix(res['y_test'], res['y_pred'])
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        
        sns.heatmap(
            cm_norm, ax=ax, annot=True, fmt='.1%',
            cmap='Blues', vmin=0, vmax=1,
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            linewidths=0.4, cbar=False,
        )
        acc = res['metrics']['accuracy']
        ax.set_title(f'{model_name}\nAcc={acc:.3f}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.tick_params(axis='x', rotation=45)
    
    fig.suptitle('Normalised Confusion Matrices (recall per class)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../results/03_confusion_matrices_all.png', bbox_inches='tight', dpi=150)
    plt.show()

except NameError:
    print('`all_results` not in memory. Run Notebook 02 first.')

---
## 9. Conclusions & Recommendations

### Architecture Summary

| Model | Params | Acc | F1 Macro | κ | Best use-case |
|-------|--------|-----|----------|---|---------------|
| **EEGNet** | ~4 K | — | — | — | Embedded / real-time BCI, few parameters |
| **ResNet1D** | ~500 K | — | — | — | Offline analysis, GPU available |
| **Bi-LSTM** | ~370 K | — | — | — | Interpretable results, temporal patterns |

### Key Findings

1. **Class imbalance** (Rest×31 vs BEO×1) strongly impacts per-class metrics.
   Inverse-frequency class weights partially mitigated this.

2. **Motor vs. Rest** (M8) is the easiest classification boundary.
   The hardest boundary is between similar limb movements (e.g., DLH vs. PLF).

3. **Cross-subject generalisation** is challenging (~chance across subjects differs).
   Subject-specific fine-tuning would improve performance significantly.

4. **EEGNet** achieves competitive performance with 1000× fewer parameters than ResNet,
   confirming its efficiency advantage for EEG BCI applications.

### Next Steps

- [ ] Fine-tune pre-trained model with few-shot subject adaptation
- [ ] Add frequency-domain features (STFT/Morlet wavelets) as input
- [ ] Ensemble EEGNet + Bi-LSTM for improved robustness
- [ ] LOSO (Leave-One-Subject-Out) cross-validation for rigorous evaluation
- [ ] Explore ShallowConvNet / DeepConvNet (Schirrmeister 2017) as baselines